# Project 9 - LLM Project: Building a News Research Tool

In [1]:
# ==============================================================================
# STEP 1: INSTALL REQUIRED PROJECT LIBRARIES
# ==============================================================================

# Install Streamlit for interactive dashboard creation
!pip install -q streamlit

# Install core LangChain framework and Groq integration package
!pip install -q langchain langchain-groq

# Install NewsAPI official Python client for fetching live news articles
!pip install -q newsapi-python

# Install python-dotenv to handle environment variables and API key security
!pip install -q python-dotenv

# Install gTTS (Google Text-to-Speech) to generate audio overviews of summaries
!pip install -q gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.29.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [2]:
# ==============================================================================
# PROJECT IMPORTS & LIBRARY BREAKDOWN
# ==============================================================================

# Core & System Utilities
import os                        # Accesses operating system environment variables safely
import io                        # Handles in-memory binary streams for audio file generation

# Streamlit Dashboard UI
import streamlit as st           # Framework for building the interactive web user interface

# Security & Secrets Management
from dotenv import load_dotenv   # Reads key-value pairs from a .env file into environment variables

# News Data Fetching
from newsapi import NewsApiClient  # Official Python wrapper to fetch news articles from NewsAPI

# LangChain & AI Orchestration
from langchain_groq import ChatGroq         # Connects LangChain directly to Groq's fast LLM engine
from langchain_core.prompts import PromptTemplate # Formats custom prompt templates for the LLM

# Audio Feature
from gtts import gTTS            # Converts output summary text into playable speech audio (MP3)

In [3]:
# ==============================================================================
# STEP 2: VERIFY SECURE ACCESS TO API KEYS
# ==============================================================================

import os
from google.colab import userdata

# Retrieve keys securely from Colab Secrets Manager
try:
    groq_key = userdata.get('GROQ_API_KEY')
    news_key = userdata.get('NEWS_API_KEY')

    # Store into OS environment variables
    os.environ["GROQ_API_KEY"] = groq_key
    os.environ["NEWS_API_KEY"] = news_key

    print("Groq API Key loaded successfully!")
    print("NewsAPI Key loaded successfully!")
    print("\n Both API keys are configured safely without exposing them in code!")

except Exception as e:
    print(" Key error:", e)
    print("Please verify secret names 'GROQ_API_KEY' and 'NEWS_API_KEY' are toggled ON in the  tab.")

Groq API Key loaded successfully!
NewsAPI Key loaded successfully!

 Both API keys are configured safely without exposing them in code!


In [14]:
# ==============================================================================
# STEP 3 FIX: UPDATE LANGCHAIN_CONFIG.PY TO SAFELY READ ENVIRONMENT VARIABLES
# ==============================================================================

%%writefile langchain_config.py
import os
import streamlit as st
from newsapi import NewsApiClient
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

def fetch_and_summarize(query, from_date=None, language='en'):
    """
    Fetches news articles via NewsAPI and summarizes them using Groq LLM.
    Returns:
        tuple: (summary_text, list_of_image_dictionaries)
    """
    # 1. Safely retrieve API keys (checks environment variables first, then Streamlit secrets)
    groq_key = os.getenv("GROQ_API_KEY") or (st.secrets.get("GROQ_API_KEY") if hasattr(st, "secrets") and "GROQ_API_KEY" in st.secrets else None)
    news_key = os.getenv("NEWS_API_KEY") or (st.secrets.get("NEWS_API_KEY") if hasattr(st, "secrets") and "NEWS_API_KEY" in st.secrets else None)

    if not news_key or not groq_key:
        raise ValueError("API Keys are missing! Ensure environment variables or Streamlit secrets are set.")

    # 2. Initialize NewsAPI client
    newsapi = NewsApiClient(api_key=news_key)

    # 3. Query articles from NewsAPI
    response = newsapi.get_everything(
        q=query,
        from_param=from_date,
        language=language,
        sort_by='relevancy',
        page_size=10
    )

    articles = response.get('articles', [])
    if not articles:
        return "No relevant news articles found for the specified query.", []

    compiled_texts = []
    images = []

    # 4. Process top 5 relevant articles
    for idx, art in enumerate(articles[:5]):
        title = art.get('title', 'No Title')
        desc = art.get('description', '') or ''
        url = art.get('url', '#')
        img_url = art.get('urlToImage')

        compiled_texts.append(f"Article {idx+1}: {title} - {desc}")

        # Collect valid image metadata for visual UI
        if img_url:
            images.append({"title": title, "url": img_url, "link": url})

    context = "\n".join(compiled_texts)

    # 5. Initialize Groq LLM model via LangChain integration
    llm = ChatGroq(
        temperature=0.3,
        model_name="llama3-8b-8192",
        groq_api_key=groq_key
    )

    # 6. Define prompt structure for equity analyst summary
    prompt = PromptTemplate(
        input_variables=["query", "context"],
        template="""
        You are an expert equity research analyst. Given the search query and the news article excerpts,
        provide a clear, executive-level summary highlighting market impact, geopolitical issues, and major takeaways.

        Query: {query}
        Article Excerpts:
        {context}
        """
    )

    # 7. Execute chain (Prompt -> Groq LLM)
    chain = prompt | llm
    result = chain.invoke({"query": query, "context": context})

    return result.content, images

Overwriting langchain_config.py


In [5]:
# ==============================================================================
# STEP 4: CREATE STREAMLIT DASHBOARD INTERFACE (app.py)
# ==============================================================================

%%writefile app.py
import streamlit as st
import os
import io
from dotenv import load_dotenv
from gtts import gTTS
from langchain_config import fetch_and_summarize

# Load environment variables if running locally or in Colab
load_dotenv()

# Page configuration
st.set_page_config(page_title="Equity Research News Tool", layout="wide", page_icon="📈")

# Main Title & Subtitle
st.title("📈 Equity Research & News Summarization Dashboard")
st.markdown("Powered by **LangChain**, **Groq LLM**, and **NewsAPI**.")

# ------------------------------------------------------------------------------
# SIDEBAR CONTROLS (Inputs, Dates, Languages)
# ------------------------------------------------------------------------------
st.sidebar.header("🔍 Search & Filter Controls")

# Input Query
query = st.sidebar.text_input("Research Topic / Query", value="Russia Ukraine war impact on market")

# Date Filter (Default set to broad search)
from_date = st.sidebar.date_input("From Date", value=None, help="Select starting date for news search")

# Language Selection
language = st.sidebar.selectbox("Language", options=["en", "es", "fr", "de"], index=0, help="Select language of news articles")

# Submit Button
fetch_button = st.sidebar.button("Fetch & Summarize", type="primary")

# ------------------------------------------------------------------------------
# MAIN CONTENT AREA
# ------------------------------------------------------------------------------
if fetch_button:
    if not query.strip():
        st.error("Please enter a valid research topic.")
    else:
        with st.spinner("Fetching news articles and generating summary with Groq..."):
            try:
                # Format date parameter
                date_str = from_date.strftime('%Y-%m-%d') if from_date else None

                # Fetch summary and image metadata from backend
                summary, images = fetch_and_summarize(query, from_date=date_str, language=language)

                # 1. Executive Summary Output
                st.subheader("💡 Executive Summary")
                st.write(summary)

                # 2. Audio Overview Feature (Google TTS)
                st.subheader("🔊 Audio Overview")
                tts = gTTS(text=summary[:500], lang=language)
                audio_fp = io.BytesIO()
                tts.write_to_fp(audio_fp)
                st.audio(audio_fp.getvalue(), format="audio/mp3")

                # 3. Export Summary Option
                st.subheader("📥 Export Summary")
                st.download_button(
                    label="Download Summary (.txt)",
                    data=summary,
                    file_name=f"{query.replace(' ', '_')}_summary.txt",
                    mime="text/plain"
                )

                # 4. Related Images & Visuals Section
                if images:
                    st.subheader("🖼️ Related News Visuals")
                    cols = st.columns(min(len(images), 3))
                    for idx, img in enumerate(images[:3]):
                        with cols[idx]:
                            st.image(img['url'], caption=img['title'], use_container_width=True)
                            st.markdown(f"[Read Full Article]({img['link']})")

            except Exception as e:
                st.error(f"An error occurred while running the research pipeline: {e}")

Writing app.py


In [10]:
# ==============================================================================
# STEP 5: CREATE REQUIREMENTS & GITIGNORE FILES
# ==============================================================================

# 1. Create requirements.txt
requirements_content = """streamlit
langchain
langchain-groq
newsapi-python
python-dotenv
gTTS
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)
print("Created requirements.txt successfully!")

# 2. Create .gitignore
gitignore_content = """.env
__pycache__/
*.pyc
.DS_Store
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)
print("Created .gitignore successfully!")

Created requirements.txt successfully!
Created .gitignore successfully!


In [11]:
!ls -a

.   app.py   .gitignore		 langchain_config.py  sample_data
..  .config  .ipynb_checkpoints  requirements.txt
